# cross-entropy-classification-loss — faded example 2: Cross-entropy with label smoothing

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `cross-entropy-classification-loss`. Running the beacon reports progress on the `Loss: Cross-entropy classification` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Loss: Cross-entropy classification` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cross-entropy-classification-loss`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cross-entropy-classification-loss"
DD_SUBTOPIC = "Loss: Cross-entropy classification"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Label smoothing replaces the one-hot target with a soft distribution: `(1 - eps)` mass on the true class and `eps / C` spread over all classes. The loss is then `-(smoothed_target * log_softmax(logits)).sum(-1).mean()`, which matches `F.cross_entropy(..., label_smoothing=eps)`.

## Faded exercise 2

### Faded — cross-entropy with label smoothing

Implement `smoothed_ce(logits, labels, eps)` matching `F.cross_entropy(logits, labels, label_smoothing=eps)`. The one-hot construction, the log-softmax, and the reduction are provided. Complete the ONE step that builds the smoothed target distribution from the one-hot tensor.

Inputs:
- `logits`: `(B, C)` raw scores.
- `labels`: `(B,)` int64 class indices.
- `eps`: float smoothing factor in `[0, 1)`.

Output: scalar loss.

**Fill in:** builds the smoothed target distribution: (1 - eps) on the true class plus eps/C uniformly over all classes

In [ ]:
import torch.nn.functional as F

def smoothed_ce(logits, labels, eps):
    C = logits.shape[-1]
    one_hot = F.one_hot(labels, num_classes=C).float()   # (B, C)
    log_probs = F.log_softmax(logits, dim=-1)            # (B, C)
    smoothed = one_hot * (1 - eps) + eps / C
    return -(smoothed * log_probs).sum(dim=-1).mean()


import torch.nn.functional as F

def _test():
    t.manual_seed(0)
    logits = t.randn(8, 6)
    labels = t.randint(0, 6, (8,))
    eps = 0.1
    mine = smoothed_ce(logits, labels, eps)
    ref = F.cross_entropy(logits, labels, label_smoothing=eps)
    assert mine.shape == (), f"expected scalar, got {mine.shape}"
    assert t.allclose(mine, ref, atol=1e-5), f"mine={mine.item()} ref={ref.item()}"


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn.functional as F

def smoothed_ce(logits, labels, eps):
    C = logits.shape[-1]
    one_hot = F.one_hot(labels, num_classes=C).float()   # (B, C)
    log_probs = F.log_softmax(logits, dim=-1)            # (B, C)
    smoothed = one_hot * (1 - eps) + eps / C
    return -(smoothed * log_probs).sum(dim=-1).mean()
```
</details>